In [ ]:
import pandas as pd
import numpy as np
import random

import sys, os
from tqdm import tqdm

sys.path.append(os.path.join(os.getcwd(), 'sql_engine_connection'))

from engine_connect import get_engine

import seaborn as sns
import matplotlib.pyplot as plt

from datetime import datetime, timedelta

In [ ]:
data = pd.read_json('data/events')
data['collector_tstamp'] = data.collector_tstamp.apply(lambda x: datetime.utcfromtimestamp(x / 1000))

In [ ]:
data.head()

,partner_key,collector_tstamp,domain_userid,ab_slot1_variant,event_name,product_domain,product_id,variant_size_label,fitpredictor_fuxp_seed_brand,fitpredictor_fuxp_seed_category,fitpredictor_fuxp_seed_size,prediction_type,prediction_size,dvce_screenwidth
0,Partner B,2017-10-21 04:23:59,7fef686a7f46988a53fc280cc713d011f762482f,None,viewed_product,Female Bottoms,T63WE,None,None,None,None,None,None,320
1,Partner A,2018-04-14 22:22:59,fd4a2cb28ba50022d6a04a36ce8c9cc9ad932eb4,Test,viewed_product,Dresses,39053,None,None,None,None,mn,XS,1280
2,Partner A,2018-04-14 19:31:08,c4f893cba554ed80df6a92b7fe004b32558fd652,Control,viewed_product,Dresses,46595,None,None,None,None,None,None,375
3,Partner A,2018-04-13 17:07:26,02f2b1ac4f5afc7a96f9fa188f00ca8bee7b2510,Test,viewed_product,Dresses,3229646,None,None,None,None,None,None,1920
4,Partner B,2017-10-16 20:25:07,4b2098df581a0378bc599dd238fe1fe648685883,None,viewed_product,Male Tops,B2AEX,None,None,None,None,None,None,375


In [ ]:
data.event_name.unique()

array(['viewed_product', 'added_variant_to_cart', 'opened_editor',
       'ordered_variant', 'selected_size', 'completed_profiling',
       'opened_brand_list', 'selected_category', 'selected_brand'],
      dtype=object)

In [ ]:
df_A = data[data.partner_key == 'Partner A']
df_B = data[data.partner_key == 'Partner B']

In [ ]:
df_A_test = df_A[df_A.ab_slot1_variant == 'Test']
df_A_control = df_A[df_A.ab_slot1_variant == 'Control']

In [ ]:
df_A_test.event_name.unique()

array(['viewed_product', 'added_variant_to_cart', 'opened_editor',
       'opened_brand_list', 'selected_size', 'ordered_variant',
       'completed_profiling', 'selected_brand', 'selected_category'],
      dtype=object)

In [ ]:
no_users_who_made_order = df_A_test[df_A_test.event_name == 'ordered_variant'].domain_userid.nunique()
no_users_who_visited_site = df_A_test.domain_userid.nunique()

conv_rate_test = no_users_who_made_order / no_users_who_visited_site
conv_rate_test

0.05135861451179457

In [ ]:
no_users_who_made_order

15308

In [ ]:
no_users_who_visited_site

298061

In [ ]:
0.05 * 298061

14903.050000000001

In [ ]:
no_users_who_made_order = df_A_control[df_A_control.event_name == 'ordered_variant'].domain_userid.nunique()
no_users_who_visited_site = df_A_control.domain_userid.nunique()

conv_rate_control = no_users_who_made_order / no_users_who_visited_site
conv_rate_control

0.05049136571359243

In [ ]:
no_users_who_made_order

14982

In [ ]:
no_users_who_visited_site

296724

In [ ]:
df_A_test.head()

,partner_key,collector_tstamp,domain_userid,ab_slot1_variant,event_name,product_domain,product_id,variant_size_label,fitpredictor_fuxp_seed_brand,fitpredictor_fuxp_seed_category,fitpredictor_fuxp_seed_size,prediction_type,prediction_size,dvce_screenwidth
1,Partner A,2018-04-14 22:22:59,fd4a2cb28ba50022d6a04a36ce8c9cc9ad932eb4,Test,viewed_product,Dresses,39053,None,None,None,None,mn,XS,1280
3,Partner A,2018-04-13 17:07:26,02f2b1ac4f5afc7a96f9fa188f00ca8bee7b2510,Test,viewed_product,Dresses,3229646,None,None,None,None,None,None,1920
6,Partner A,2018-04-12 03:07:59,f3054200e32843bdbf56bb507e0a6feecac69cc6,Test,viewed_product,Dresses,6521070,None,None,None,None,None,None,1440
9,Partner A,2018-04-14 02:32:24,ea74e0421693e37e67081dd5471ad2cca4a47604,Test,added_variant_to_cart,Dresses,92779,XL,None,None,None,None,None,375
10,Partner A,2018-04-14 09:10:56,cdce2e40516c8fe894fc09bcb8673de350479986,Test,viewed_product,Dresses,4648072,None,None,None,None,None,None,768


In [ ]:
# calculate variance of conversion rate

users = df_A_test.domain_userid.unique()

# values of the random variable which equals 1 if given user (no matter which, every user has the same distribution) bought any product or 0 otherwise
user_variable_values = []
data_ordered_products = df_A_test[df_A_test.event_name == 'ordered_variant']

for user in tqdm(users):
    df = data_ordered_products[data_ordered_products.domain_userid == user]
    
    if len(df) > 0:
        user_variable_values.append(1)
    else:
        user_variable_values.append(0)

100%|█████████████████████████████████████████████████████████████████████████| 298061/298061 [19:15<00:00, 257.89it/s]


In [ ]:
conv_rate_variance = np.var(user_variable_values) / len(users)
conv_rate_variance

1.6345951743845539e-07

In [ ]:
np.mean(user_variable_values)

0.05135861451179457

In [ ]:
np.var(user_variable_values)

0.04872090722722345